<a href="https://colab.research.google.com/github/Yugansh-Varshney/ANN-Classification-Churn-Prediction-Model/blob/main/DL_Lab1_Data_to_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DL Lab 1 — From Cleaned Data to a Single ANN Forward Pass

**Goal of today:** you already know *what* an ANN is on paper. Today we make it real:
1. Load a real messy dataset and clean it (recap + practice)
2. See exactly how a cleaned row of data becomes an "input vector"
3. Build ONE forward pass through a tiny ANN by hand using NumPy — no PyTorch, no training yet
4. Understand `output = activation(W·x + b)` with real numbers

We are **not training** anything today. No backpropagation, no loss minimization. That's next lab.
Today is about making the architecture diagram feel like real numbers flowing through a real network.


## Step 1 — Setup (10–15 min)
Run this in Google Colab (colab.research.google.com) — no installs needed.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("numpy:", np.__version__)
print("pandas:", pd.__version__)


numpy: 2.0.2
pandas: 2.2.2


### Instructor talking points (2 min)
- We're using **NumPy + Pandas only** today. No deep learning framework yet.
- Why? Because you should see the matrix multiplication and the math before a framework hides it from you.
- Next lab, we introduce PyTorch and actually train this network.


## Step 2 — Load the Dataset (5 min)
We'll use the **Titanic dataset** — it's small, famous, and (importantly) *messy* — perfect for practicing cleaning.
Each row = one passenger. Target: did they survive (1) or not (0)?


In [2]:
import seaborn as sns

df = sns.load_dataset('titanic')
df.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


### Instructor talking points (3 min)
Ask the class: *"What do you see wrong with this data already?"*
Let them look before you say anything. Expected answers: missing values, text columns (sex, embarked),
mixed types, columns that seem useless (deck, embark_town duplicate of embarked).


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [4]:
df.isnull().sum().sort_values(ascending=False)


,0
deck,688
age,177
embarked,2
embark_town,2
sex,0
pclass,0
survived,0
fare,0
parch,0
sibsp,0


## Step 3 — Data Cleaning (35–40 min, hands-on)
This is the part you already taught in theory. Now they do it themselves, column by column.

**Have students do each of the following one at a time — pause after each and ask what changed.**


### 3.1 — Drop columns that are not useful as ANN inputs (redundant or too many missing)

In [5]:
# 'deck' has too many missing values, 'embark_town'/'alive'/'class' duplicate other columns,
# 'who'/'adult_male' duplicate 'sex'/'age' info. Keep it simple for a first lab.
df_clean = df.drop(columns=['deck', 'embark_town', 'alive', 'class', 'who', 'adult_male', 'alone'])
df_clean.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


### 3.2 — Handle missing values

In [6]:
# 'age' has missing values -> fill with median (robust to outliers)
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())

# 'embarked' has 2 missing values -> fill with the most common port (mode)
df_clean['embarked'] = df_clean['embarked'].fillna(df_clean['embarked'].mode()[0])

df_clean.isnull().sum()


,0
survived,0
pclass,0
sex,0
age,0
sibsp,0
parch,0
fare,0
embarked,0


### 3.3 — Encode categorical (text) columns as numbers
ANNs only understand numbers. `sex` and `embarked` are text -> must be converted.


In [7]:
# sex: binary -> simple 0/1 mapping
df_clean['sex'] = df_clean['sex'].map({'male': 0, 'female': 1})

# embarked: 3 categories (C, Q, S) -> one-hot encoding (no fake ordering implied)
df_clean = pd.get_dummies(df_clean, columns=['embarked'], drop_first=False)

df_clean.head()


,survived,pclass,sex,age,sibsp,parch,fare,embarked_C,embarked_Q,embarked_S
0,0,3,0,22.0,1,0,7.2500,False,False,True
1,1,1,1,38.0,1,0,71.2833,True,False,False
2,1,3,1,26.0,0,0,7.9250,False,False,True
3,1,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,0,35.0,0,0,8.0500,False,False,True


### Instructor talking point
Ask: *"Why didn't we map embarked to 0/1/2 like we did with sex?"*
Answer: sex has only 2 categories, so 0/1 is fine. Embarked has 3 unrelated categories —
mapping to 0/1/2 would falsely imply an order (like Q > C). One-hot avoids that.


### 3.4 — Scale numeric features
ANNs train better when input numbers are on a similar scale. We'll standardize manually (no sklearn) so the math is visible.

In [8]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Select the columns to scale
columns_to_scale = ['age', 'fare']

# Apply standardization to the selected columns
df_clean[columns_to_scale] = scaler.fit_transform(df_clean[columns_to_scale])

df_clean[columns_to_scale].describe()

,age,fare
count,8.910000e+02,8.910000e+02
mean,2.272780e-16,3.987333e-18
std,1.000562e+00,1.000562e+00
min,-2.224156e+00,-6.484217e-01
25%,-5.657365e-01,-4.891482e-01
50%,-1.046374e-01,-3.573909e-01
75%,4.333115e-01,-2.424635e-02
max,3.891554e+00,9.667167e+00


### 3.5 — Final check
By now every column should be numeric with no missing values — ready to feed into an ANN.


In [9]:
df_clean.info()
df_clean.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   survived    891 non-null    int64  
 1   pclass      891 non-null    int64  
 2   sex         891 non-null    int64  
 3   age         891 non-null    float64
 4   sibsp       891 non-null    int64  
 5   parch       891 non-null    int64  
 6   fare        891 non-null    float64
 7   embarked_C  891 non-null    bool   
 8   embarked_Q  891 non-null    bool   
 9   embarked_S  891 non-null    bool   
dtypes: bool(3), float64(2), int64(5)
memory usage: 51.5 KB


,survived,pclass,sex,age,sibsp,parch,fare,embarked_C,embarked_Q,embarked_S
0,0,3,0,-0.565736,1,0,-0.502445,False,False,True
1,1,1,1,0.663861,1,0,0.786845,True,False,False
2,1,3,1,-0.258337,0,0,-0.488854,False,False,True
3,1,1,1,0.433312,1,0,0.420730,False,False,True
4,0,3,0,0.433312,0,0,-0.486337,False,False,True


## Step 4 — Bridging Data to ANN Architecture (10–15 min)
This is the key conceptual moment of the lab.

- Each **row** of `df_clean` (minus the target column `survived`) = **one input vector `x`**
- Each **feature/column** = **one input neuron**
- The number of remaining columns = **size of the input layer**


In [10]:
X = df_clean.drop(columns=['survived']).values.astype(float)
y = df_clean['survived'].values.astype(float)

print("Shape of X (samples, features):", X.shape)
print("So the input layer of our ANN needs", X.shape[1], "neurons.")
print("\nFirst input vector x (one passenger):\n", X[0])


Shape of X (samples, features): (891, 9)
So the input layer of our ANN needs 9 neurons.

First input vector x (one passenger):
 [ 3.          0.         -0.56573646  1.          0.         -0.50244517
  0.          0.          1.        ]


### Instructor talking point
Draw this on the board right now:
`[input vector, one number per feature] -> [input layer, one neuron per number]`
This is the "aha" moment: the diagram they've seen in theory IS this row of numbers.


## Step 5 — Manual Forward Pass with NumPy (30–35 min, hands-on)
We'll build ONE hidden layer + ONE output neuron completely by hand.
No training yet — just: given random weights, what comes out the other end?


### 5.1 — Forward pass for a single sample

In [12]:
from scipy.special import expit as sigmoid

n_features = X.shape[1]
n_hidden = 4   # you choose this it's a design decision, not derived from data

np.random.seed(42)
W1 = np.random.randn(n_hidden, n_features) * 0.1   # hidden layer weights
b1 = np.zeros(n_hidden)                             # hidden layer bias

x = X[0]                       # take the first passenger as our input vector
z1 = W1 @ x + b1               # linear step: W . x + b
a1 = sigmoid(z1)               # activation step

print("z1 (before activation):", z1)
print("a1 (after activation): ", a1)

z1 (before activation): [ 0.22949179  0.33140467 -0.42141552 -0.05236528]
a1 (after activation):  [0.55712246 0.58210111 0.39617808 0.48691167]


- `W1 @ x` is literally what "fully connected layer" means: every input neuron connects to every hidden neuron.
- `W1` shape is `(n_hidden, n_features)` — each row of W1 is the weights feeding into ONE hidden neuron.
- `b1` shifts the result before squashing.
- `sigmoid` squashes any number into (0, 1) — this is the "activation."
- Right now the weights are **random** — the network knows nothing yet. That's expected. Training (next lab) is the process of adjusting W and b so the output becomes meaningful.


### 5.2 — Add the output layer (1 neuron, since this is binary classification: survived or not)

In [13]:
n_output = 1

W2 = np.random.randn(n_output, n_hidden) * 0.1
b2 = np.zeros(n_output)

z2 = W2 @ a1 + b2
a2 = sigmoid(z2)

print("Final output (predicted probability of survival):", a2)
print("Actual label for this passenger:", y[0])


Final output (predicted probability of survival): [0.46369631]
Actual label for this passenger: 0.0


### Instructor talking point
Point out: the prediction is basically random right now (weights are random) — compare it to the actual label
and note it's probably wrong. Ask: *"How would the network learn to get this right?"* → that's next lab
(loss function + backpropagation + gradient descent). Today's job was just to see the plumbing work.


### 5.3 — Vectorize: do this for ALL passengers at once (not just one)
This is how it's actually done in practice — one matrix multiply instead of a loop.

In [14]:
Z1 = X @ W1.T + b1        # shape: (n_samples, n_hidden)
A1 = sigmoid(Z1)

Z2 = A1 @ W2.T + b2       # shape: (n_samples, 1)
A2 = sigmoid(Z2)

print("A2 shape:", A2.shape)
print("First 5 predicted probabilities:\n", A2[:5].flatten())
print("First 5 actual labels:\n", y[:5])


A2 shape: (891, 1)
First 5 predicted probabilities:
 [0.46369631 0.46922911 0.46512913 0.4673243  0.46316004]
First 5 actual labels:
 [0. 1. 1. 1. 0.]


### Instructor talking point
This one cell just ran a full forward pass for **all ~891 passengers simultaneously** — this is why
matrix math matters for ANNs: it's not a loop over samples, it's one matrix multiplication.


## Step 6 — Wrap-up & Recap (10–15 min)

Walk through this chain out loud with the class, pointing at the code above for each step:

`raw messy data -> cleaned numeric data -> one row = input vector -> W1,b1 -> hidden layer -> activation -> W2,b2 -> output neuron -> activation -> prediction`

**Ask the class (discussion, no need to answer today):**
- Why were the predictions basically random / wrong?
- What do you think needs to change so predictions get better?
- What do W1 and W2 represent, physically?

**Next lab preview:** we'll introduce a *loss function* (how wrong are we?) and *gradient descent*
(how do we nudge W and b to be less wrong?) — using PyTorch instead of manual NumPy.

**Optional take-home exercise:** change `n_hidden` from 4 to 8 or 2, rerun Step 5, and see the output
shape doesn't change (why?) but the numbers do (why?).
